# Notebook for calculating the integrated change of length data for Rufu & Canup (2021)

## Preamble

### Load required modules

In [1]:
import numpy as np
import scipy as sp
import sys
import os
import struct
from scipy import constants as const

import time as tclock

from scipy.signal import savgol_filter

#package to use wildcards 
import fnmatch

import csv

from scipy.interpolate import LinearNDInterpolator
from scipy.interpolate import griddata

from scipy import interpolate

#plotting packages
import matplotlib as mpl
import matplotlib.pyplot as plt
import pylab
import matplotlib.cm as cm
from matplotlib import gridspec


cwd = os.getcwd()
print(cwd)
if sys.platform== 'darwin':
    sys.path.insert(0, cwd+'/Support_scipts')
    print(cwd+'/Support_scripts')
elif (sys.platform== 'win32') | (sys.platform== 'win64'):
    sys.path.insert(0,cwd+"\\Support_scipts")
    
#HERCULES_structures
from HERCULES_structures import *
from surface_size_calc import *
from HERCULES_random_planet_database_structure_1D import *

#functions for calculating non-evenly spaced numerical differentials
from gradients import *

#import colormaps
import colormaps as cmaps
import matplotlib.cm as cm

import svglib.svglib as svglib
svglib.register_font('helvetica', './Helvetica.ttc')

/Users/vq21447/Documents/Lock_2026_SI
/Users/vq21447/Documents/Lock_2026_SI/Support_scripts
CHECK THE LOCATION OF ODYSSEY BACKUP
CHECK THE LOCATION OF ODYSSEY BACKUP


('helvetica', True)

### Constants

In [2]:

#CONSTANTS
MEarth=5.972E24
LEM=3.5E34
REarth=6.371E6
MMoon=7.34767309E22

aMoon=0.3844E9
aCassini=30*REarth
aRoche=2.9*REarth

#for HERCULES
MEarth_H=5.9879648E24
LEM_H=3.53E34

### Parameters

In [3]:
#PARAMS

#info for HERCULES arrays
Hdir='Earth_correct_params_S3.20c'
Hname='Earth_correct_params_S3.20c'

#Earth's moment of inertia used to convert to AM
C_Earth=0.335

#which plot do you want
#0: A=5, tau=49E5, Q/k=406
#1: A=500, tau=200E5, Q/k=406
#2: A=10000, tau=1500E5, Q/k=406
flag_data=0

#Give data files (for all cases)
dir='Data/Rufu&Canup_2020_quasi_resonance/ResultsEvection_2'
file_names=['EvolutionA5_Tau49.0e5_Phi0.txt',
            'EvolutionA500_Tau200.0e5_Phi0.txt',
            'EvolutionA10000_Tau1500.0e5_Phi0.txt']

#output file root
data_output_file_root='Rufu&Canup_surf_change'

#directory to put procesed length data
data_dir='Data/Rufu&Canup_2020_quasi_resonance'

#whether to overwrite saved data or make a new file
flag_overwrite=0


## Main

### Load in HERCULES database

In [4]:

#MAIN

########################################################################################
#read in the database

Hdatabase=HERCULES_random_planet_database_1D()
Hdatabase.make_array(Hdir,Hname)
Hdatabase.initialize_interpolation([0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],\
                                   [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]], flag_extrap=1)

#extract the latitudes for each Mu point
Nmu=Hdatabase.parr[0].Nmu
lat=np.arccos(Hdatabase.parr[0].layers[0].mu)*180/np.pi


Earth_correct_params_S3.20c
	 Earth_correct_params_S3.20c_AM3.3125
	 Earth_correct_params_S3.20c_AM3.30625
	 Earth_correct_params_S3.20c_AM3.325


### Read in the orbital data

In [5]:

fdata=open(dir+'/'+file_names[flag_data])

reader = csv.reader(fdata, delimiter="\t", skipinitialspace=True)
temp = list(reader)

Ctime=[]
CkQ=[]
Ca=[]
Comg=[]
Comg_moon=[]
Ce=[]
Cphi=[]


for i in np.arange(len(temp)):
    
    if flag_data<8:
        if i==0:
            Aset=temp[i][0]
            Aset=float(Aset[6:])
            print(Aset)

        elif i==1:
            tau_set=temp[i][0]
            tau_set=float(tau_set[13:])
            print(tau_set)

        elif (i>3)&(np.size(temp[i])>3):

            Ctime.append(temp[i][0]) #time
            Comg.append(temp[i][1]) #rotation rate of Earth
            Comg_moon.append(temp[i][2]) #rotation rate of Moon
            Ce.append(temp[i][3]) #eccentricity
            Ca.append(temp[i][4]) #semi-major axis
            Cphi.append(temp[i][5]) #Phi parameter related to tidal lag
    elif (flag_data==8)|(flag_data==9):
        Aset='Zahnle2015'
        if flag_data==8:
            tau_set='highQ'
        if flag_data==9:
            tau_set='lowQ'
            
        if (i>1)&(np.size(temp[i])>3):

            Ctime.append(temp[i][0]) #time
            CkQ.append(temp[i][1]) #forced tidal params
            Comg.append(temp[i][2]) #rotation rate of Earth
            Comg_moon.append(temp[i][3]) #rotation rate of Moon
            Ce.append(temp[i][4]) #eccentricity
            Ca.append(temp[i][5]) #semi-major axis
            Cphi.append(temp[i][6]) #Phi parameter related to tidal lag
        
        
#convert to useful type and units
omg_norm=np.sqrt(const.G*MEarth/(REarth**3))
Ctime=np.asarray(Ctime, dtype=np.float64)*1E4
Ca=np.asarray(Ca, dtype=np.float64)*REarth
Ce=np.asarray(Ce, dtype=np.float64)
Comg=np.asarray(Comg, dtype=np.float64)*omg_norm
Comg_moon=np.asarray(Comg, dtype=np.float64)*omg_norm

#calculate the AM and de-normalize
CL_tot=Comg/omg_norm+1.07E-3*Comg_moon/omg_norm+0.0367*np.sqrt(Ca/REarth*(1-Ce**2))
CL_tot=CL_tot*C_Earth*MEarth*REarth**2*omg_norm

CL=Comg*C_Earth*MEarth*REarth**2

print(CL[0]/LEM)
     

print('done')

5.0
4900000.0
2.0249183381015543
done


In [6]:
#define the output files
#define an output data file
if flag_overwrite==1:
    overwrite=''
else:
    overwrite='_recalc'
    
# data_output_file=data_dir+'/'+data_output_file_root+'_A_'+str(Aset)+'_tau_'+str(tau_set/1E5)+'E5.bin'
data_output_file_int=data_dir+'/'+data_output_file_root+'_max_integrated_A_'+str(Aset)+'_tau_'+str(tau_set/1E5)+'E5'+overwrite+'.bin'

In [7]:
#Calcualte the change of surface at a selection of time points
print('begin')
tstart=tclock.time()

#Whether or not to overwrite
flag_overwrite=1

#read in the file and loop over each data point
if (os.path.isfile(data_output_file_int)==False)|(flag_overwrite==1):
    #open the output file
    dataf_int=open(data_output_file_int, "wb")
else:
    dataf_int=open(data_output_file_int, "ab")

checkpoints=np.linspace(1,501,1001)
# steps=np.arange(0,30000)
steps=np.arange(0,np.size(Ctime))
Nt=np.size(steps)
time=Ctime[steps]

if steps[0]>=np.size(Ctime):
    print('Too many!!!')
    print(ldkjfsl)


dLdt=gradient2(Ctime,CL)

print(Nt, np.size(Ctime))


count=-1
for i in steps[0:]:
    count+=1
    #print(i, np.size(steps),count)
        
    if ((i-steps[0])*1.0/Nt*100)>checkpoints[0]:
        print((i-steps[0])*1.0/Nt*100, '%')
        checkpoints=checkpoints[1:]
    
    temp_data=Hdatabase.interp_database(CL[i],[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1],\
                                       [0,0,[0],0,0,0,0,[0],[0],[0],[0],[0],[0],[0],[0],[0],[0]],flag_extrap=1)

    temp=np.asarray(temp_data[16])

    dl_lat=temp[:,0]
    dl_lon=temp[:,1]
    dA=temp[:,2]
    
    ddl_lat_dL=temp[:,3]
    ddl_lon_dL=temp[:,4]
    ddA_dL=temp[:,5]
    
    ddl_lat_dt=temp[:,3]*dLdt[i]
    ddl_lon_dt=temp[:,4]*dLdt[i]
    ddA_dt=temp[:,5]*dLdt[i]

    #longitudinal is easy. Accomodated around minor circle
    ddl_lon_dt_max=np.amax(ddl_lon_dt)*2*np.pi
    ddl_lon_dt_min=np.amin(ddl_lon_dt)*2*np.pi


    #latitudinal is harder. Need to integrate over surface
    temp=np.where(ddl_lat_dt<0)[0][-1]

    #print(lat[0:temp])
    ddl_lat_dt_min=2*integrate.trapz(ddl_lat_dt[0:temp]*np.pi/180.0,90.0-lat[0:temp])
    ddl_lat_dt_max=integrate.trapz(ddl_lat_dt[(temp+1):]*np.pi/180.0,90.0-lat[(temp+1):])
    
    #now print out these points to the file
    np.asarray(steps[i]).astype('float64').tofile(dataf_int)
    np.asarray(time[i]).astype('float64').tofile(dataf_int)
    np.asarray(ddl_lon_dt_max).astype('float64').tofile(dataf_int)
    np.asarray(ddl_lon_dt_min).astype('float64').tofile(dataf_int)
    np.asarray(ddl_lat_dt_min).astype('float64').tofile(dataf_int)
    np.asarray(ddl_lat_dt_max).astype('float64').tofile(dataf_int)

dataf_int.close()


print('end')
print('time taken',tclock.time()-tstart)
print((tclock.time()-tstart)/(1.*count)*np.size(Ctime))
    

begin
400000 400000
1.0002499999999999 %
1.50025 %
2.00025 %
2.50025 %
3.0002500000000003 %
3.5000000000000004 %
4.00025 %
4.50025 %
5.000249999999999 %
5.50025 %
6.00025 %
6.50025 %
7.000000000000001 %
7.50025 %
8.000250000000001 %
8.50025 %
9.00025 %
9.500250000000001 %
10.00025 %
10.50025 %
11.000250000000001 %
11.50025 %
12.00025 %
12.50025 %
13.00025 %
13.50025 %
14.000000000000002 %
14.500250000000001 %
15.000250000000001 %
15.50025 %
16.000249999999998 %
16.50025 %
17.00025 %
17.50025 %
18.00025 %
18.500249999999998 %
19.000249999999998 %
19.50025 %
20.00025 %
20.50025 %
21.00025 %
21.50025 %
22.000249999999998 %
22.50025 %
23.00025 %
23.50025 %
24.00025 %
24.50025 %
25.00025 %
25.50025 %
26.00025 %
26.500249999999998 %
27.000249999999998 %
27.500000000000004 %
28.000000000000004 %
28.500249999999998 %
29.00025 %
29.50025 %
30.00025 %
30.50025 %
31.00025 %
31.50025 %
32.00025 %
32.50025 %
33.000249999999994 %
33.50025 %
34.00025 %
34.50025 %
35.00025 %
35.50025 %
36.00025 %
36.5

### Compare to original data (if file not overwritten)

In [12]:
#If not overwritten can compare to previous binary file
if flag_overwrite==0:
    data_output_file_int_og=data_dir+'/'+data_output_file_root+'_max_integrated_A_'+str(Aset)+'_tau_'+str(tau_set/1E5)+'E5.bin'
    
    #read in the max deformation data 
    dataf_int = open(data_output_file_int, "rb")
    dataf_int_og = open(data_output_file_int_og, "rb")
    
    #read in the file as one massive array
    data = np.fromfile(dataf_int, dtype=np.float64, count=-1)
    dataf_int.close()
    
    data_og = np.fromfile(dataf_int_og, dtype=np.float64, count=-1)
    dataf_int_og.close()
    
    print('Maximum abs and rel difference:', np.max(np.abs(data-data_og)),np.max(np.abs((data-data_og)/(data+1E-14))))


Maximum abs and rel difference: 1.9895196601282805e-13 4.018410707119816e-15
